<a href="https://colab.research.google.com/github/diseasemodeling/MIE_525_625/blob/main/Lectures/4_MLP_Heads/4c_LLM_MLP_SHAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SENTIMENT ANALYSIS - CLASSIFY SENTIMENT AS POSITIVE or NEGATIVE
Source:
https://huggingface.co/datasets/stanfordnlp/imdb

Note: Code generated with AI prompting

In [ ]:
!pip install -q torch transformers datasets shap scikit-learn pandas numpy

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import shap
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from transformers import AutoTokenizer, AutoModel

# Set random seed for PyTorch reproducibility
torch.manual_seed(40)
np.random.seed(40)

In [ ]:
# =====================================================================
# LOAD DATASET FROM HUGGING FACE
# =====================================================================
print("📥 Fetching the official IMDb dataset from Hugging Face Hub...")
raw_dataset = load_dataset("stanfordnlp/imdb", split="train")
df = raw_dataset.to_pandas()

# Small sample for quick CPU execution
df = df.sample(n=100, random_state=40).reset_index(drop=True)
X_texts = df['text'].tolist()
y = df['label'].values


📥 Fetching the official IMDb dataset from Hugging Face Hub...


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

...

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:

# =====================================================================
# 2. Convert text data to inputs for MLP: FEATURE EXTRACTION (LLM Embeddings) USING LLM Encoder -
# =====================================================================
print("🤖 Loading DistilBERT transformer layers...")
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModel.from_pretrained(model_name)

def get_embeddings(text_list):
    embeddings = []
    llm_model.eval()
    with torch.no_grad():
        for text in text_list:
            inputs = tokenizer(text, padding=True, truncation=True, max_length=128, return_tensors="pt")
            outputs = llm_model(**inputs)
            cls_repr = outputs.last_hidden_state[0, 0, :].numpy()
            embeddings.append(cls_repr)
    return np.array(embeddings)

print("⚡ Converting strings into numeric LLM vector states...")
X_embeddings = get_embeddings(X_texts)

# Split datasets
X_train_emb, X_test_emb, y_train, y_test = train_test_split(
    X_embeddings, y, test_size=0.2, random_state=42
)
_, X_test_txt, _, _ = train_test_split(
    X_texts, y, test_size=0.2, random_state=42
)

# Convert arrays into PyTorch Tensors
X_train_tensor = torch.tensor(X_train_emb, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_emb, dtype=torch.float32)


🤖 Loading DistilBERT transformer layers...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⚡ Converting strings into numeric LLM vector states...


In [ ]:
print(X_train_tensor.shape)

In [ ]:
# =====================================================================
#  DEFINE PYTORCH MULTI-LAYER PERCEPTRON (MLP)
# =====================================================================
class PyTorchMLP(nn.Module):
    def __init__(self, input_dim):
        super(PyTorchMLP, self).__init__()
        # DistilBERT outputs 768 dimensions
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid() # Squashes output between 0 and 1 for binary classification
        )

    def forward(self, x):
        return self.network(x)

# Instantiate the model, optimizer, and Binary Cross Entropy Loss function
mlp = PyTorchMLP(input_dim=768)
criterion = nn.BCELoss()
optimizer = optim.Adam(mlp.parameters(), lr=0.005)

In [ ]:
# =====================================================================
# TRAINING THE PYTORCH MLP
# =====================================================================
print("🧠 Training the custom PyTorch MLP classifier...")
mlp.train()
epochs = 50

for epoch in range(epochs):
    optimizer.zero_grad()                  # Clear accumulated gradients
    predictions = mlp(X_train_tensor)      # Forward Pass
    loss = criterion(predictions, y_train_tensor) # Calculate Loss
    loss.backward()                        # Backward Pass (Backprop)
    optimizer.step()                       # Update Weights

    if (epoch + 1) % 10 == 0:
        print(f"  Epoch [{epoch+1}/{epochs}] | Loss: {loss.item():.4f}")

🧠 Training the custom PyTorch MLP classifier...
  Epoch [10/50] | Loss: 0.4429
  Epoch [20/50] | Loss: 0.1421
  Epoch [30/50] | Loss: 0.0170
  Epoch [40/50] | Loss: 0.0024
  Epoch [50/50] | Loss: 0.0007


In [ ]:
# =====================================================================
# INTEGRATE SHAP WITH PYTORCH PIPELINE
# =====================================================================
print("\n🔍 Building the game-theoretic SHAP explanation layer...")

def custom_model_prediction_pipeline(texts_input):
    """
    Wrapper function for SHAP:
    Takes a text list -> gets embeddings -> converts to PyTorch tensor ->
    passes through PyTorch MLP -> outputs a 2D probability matrix [P(Neg), P(Pos)]
    """
    mlp.eval()
    with torch.no_grad():
        # 1. Transform text via LLM
        embeddings = get_embeddings(texts_input)
        embeddings_tensor = torch.tensor(embeddings, dtype=torch.float32)

        # 2. Forward pass through PyTorch MLP
        pos_prob = mlp(embeddings_tensor).numpy() # Probability of being positive class
        neg_prob = 1.0 - pos_prob                 # Probability of being negative class

        # Stack horizontal layout array columns to match SHAP's expected output format
        return np.hstack((neg_prob, pos_prob))

# Build standard text explainer using the custom wrapped PyTorch function
explainer = shap.Explainer(custom_model_prediction_pipeline, tokenizer)

print("🧮 Calculating attribution maps for test instances...")
sample_reviews_to_explain = X_test_txt[:2]
shap_values = explainer(sample_reviews_to_explain)



🔍 Building the game-theoretic SHAP explanation layer...
🧮 Calculating attribution maps for test instances...


  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer:  50%|█████     | 1/2 [00:00<?, ?it/s]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 3it [01:37, 48.94s/it]               


In [ ]:
# =====================================================================
# EVALUATION AND INTERPRETABILITY OUTPUTS
# =====================================================================
mlp.eval()
with torch.no_grad():
    raw_preds = mlp(X_test_tensor).numpy()
    y_pred = (raw_preds > 0.5).astype(int).flatten()

print(f"\n🎯 PyTorch MLP Model Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

print("\n=== SHAP Local Interpretability Report ===")
for idx, text in enumerate(sample_reviews_to_explain):
    print(f"\n--- Review Sample #{idx + 1} ---")
    print(f"Review Text: \"{text[:150]}...\"")

    token_attributions = shap_values[idx, :, 1] # Check positive class attributions
    sorted_indices = np.argsort(np.abs(token_attributions.values))[::-1]

    count = 0
    for s_i in sorted_indices:
        token_string = token_attributions.data[s_i]
        impact_value = token_attributions.values[s_i]

        if token_string.strip():
            direction = "🟢 Pushing POSITIVE" if impact_value > 0 else "🔴 Pushing NEGATIVE"
            print(f"  * Token: '{token_string}' | Impact: {impact_value:+.4f} ({direction})")
            count += 1
        if count >= 3:
            break



🎯 PyTorch MLP Model Accuracy: 75.00%
              precision    recall  f1-score   support

    Negative       1.00      0.58      0.74        12
    Positive       0.62      1.00      0.76         8

    accuracy                           0.75        20
   macro avg       0.81      0.79      0.75        20
weighted avg       0.85      0.75      0.75        20


=== SHAP Local Interpretability Report ===

--- Review Sample #1 ---
Review Text: "I gave this film 10 not because it is a superbly consistent movie, but for it's pure ability to evoke emotions in its audience. The story of one-woman..."
  * Token: 'I ' | Impact: -0.0025 (🔴 Pushing NEGATIVE)
  * Token: 'hated ' | Impact: -0.0022 (🔴 Pushing NEGATIVE)
  * Token: 'find ' | Impact: -0.0020 (🔴 Pushing NEGATIVE)

--- Review Sample #2 ---
Review Text: "<br /><br />I watched this movie just a little while ago and I found that this movie was terrible! It moved very slowly and was hardly entertaining!<b..."
  * Token: '!' | Impact: -0.1

## VERSION 1 above: The above model used Pytorch for MLP.
## VERSION 2 below- uses scikit-learn MLP classifier

#### it is important to know the setting of diffferent packages. Example for MLP classified: does it output logit or probabilities?

IN Pytorch (VERSION 1 above), for binary classification problems, you can choose to
* include 'sigmoid' in final layer and use BCELoss; in this case the forward pass outputs probabilities
* Or directly use BCEWithLogitsLoss (without using sigmoid in final layer). in this case the forward pass outputs logits

In MLP calssifier (VERSION 2 below): you use mlp.predict_proba() to get probabilities.



In [ ]:
#NO need to reimport if already done before
#import numpy as np
#import pandas as pd
#import torch
#import shap
#from datasets import load_dataset
#from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
#from transformers import AutoTokenizer, AutoModel

# Initialize SHAP visualization interface for notebooks/consoles
shap.initjs()

In [ ]:
# NO need to rerun- the LLM part is the smae for both VERSIONS.
'''
# =====================================================================
# LOAD HUGGING FACE DATASET
# =====================================================================
print("📥 Fetching the official IMDb dataset from Hugging Face Hub...")
raw_dataset = load_dataset("stanfordnlp/imdb", split="train")
df = raw_dataset.to_pandas()

# Sample 100 rows for lightning-fast presentation execution on standard CPU laptops
df = df.sample(n=100, random_state=42).reset_index(drop=True)
X_texts = df['text'].tolist()
y = df['label'].values

# =====================================================================
# Use LLM Encoder to convert datasets for MLP inputs
# =====================================================================
print("🤖 Loading DistilBERT transformer layers...")
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_embeddings(text_list):
    """BATCH OPTIMIZED: Passes multiple strings through the LLM simultaneously."""
    model.eval()
    embeddings = []

    # Process inputs in micro-batches to avoid memory overflow but keep CPU speed high
    batch_size = 16
    with torch.no_grad():
        for i in range(0, len(text_list), batch_size):
            batch_texts = text_list[i:i+batch_size]
            # Essential adjustments for handling SHAP's empty string masks safely
            batch_texts = [t if (t and t.strip()) else "" for t in batch_texts]

            inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
            outputs = model(**inputs)

            # Extract [CLS] tokens for the entire batch at once
            cls_reprs = outputs.last_hidden_state[:, 0, :].numpy()
            embeddings.append(cls_reprs)

    return np.vstack(embeddings) if embeddings else np.empty((0, 768))

print("⚡ Converting raw review strings into numeric LLM vector states...")
X_embeddings = get_embeddings(X_texts)

# FIX: Properly collect y_train and y_test from your split operation
X_train_emb, X_test_emb, y_train, y_test = train_test_split(
    X_embeddings, y, test_size=0.2, random_state=42
)
_, X_test_txt, _, _ = train_test_split(
    X_texts, y, test_size=0.2, random_state=42
)
'''

In [ ]:
# Train MLP head classifier
print("🧠 Firing up and training the Multi-Layer Perceptron architecture...")
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', max_iter=300, random_state=42)
# FIX: Train using 'y_train' instead of 'y[:80]' to prevent label mismatch
mlp.fit(X_train_emb, y_train)

In [ ]:
# =====================================================================
# SETUP FOR INTERPRETATION OF MLP OUTPUTS VIA SHAP TEXT EXPLAINER
# =====================================================================
print("\n🔍 Building the game-theoretic SHAP explanation layer...")

def custom_model_prediction_pipeline(texts_input):
    """
    An end-to-end wrapper function required by SHAP.
    Takes raw list of text strings -> outputs prediction probabilities from the MLP head.
    """
    # Force input elements to standard string formatting to safely handle SHAP's underlying array conversion
    texts_input = [str(t) for t in texts_input]
    embeddings = get_embeddings(texts_input)
    return mlp.predict_proba(embeddings)

# Build an explicit text masker wrapper to guarantee consistent token decoding behavior
masker = shap.maskers.Text(tokenizer)
explainer = shap.Explainer(custom_model_prediction_pipeline, masker=masker)

# Calculate SHAP values for a few test samples
print("🧮 Calculating attribution maps for test instances...")
sample_reviews_to_explain = X_test_txt[:3]
shap_values = explainer(sample_reviews_to_explain)


In [ ]:
#THIS PART IS SAME FOR BOTH VERSIONS (as this related to use of SHAP for feature attribution)
# =====================================================================
#  APPLY SHAP EXPLAINER
# =====================================================================
print("\n=== XAI Model Interpretability Report ===")
for idx, text in enumerate(sample_reviews_to_explain):
    print(f"\n--- Review Sample #{idx + 1} ---")
    #print(f"Review Text: \"{text[:180]}...\"")
    print(f"Review Text: \"{text}...\"")#print full text
    print("Top driving tokens affecting this prediction according to SHAP:")

    # Class index 1 corresponds to "Positive" sentiment
    token_attributions = shap_values[idx, :, 1]
    sorted_indices = np.argsort(np.abs(token_attributions.values))[::-1]

    count = 0
    for s_i in sorted_indices:
        token_string = token_attributions.data[s_i]
        impact_value = token_attributions.values[s_i]

        if token_string and token_string.strip():
            direction = "🟢 Pushing POSITIVE" if impact_value > 0 else "🔴 Pushing NEGATIVE"
            print(f"  * Token: '{token_string}' | Impact: {impact_value:+.4f} ({direction})")
            count += 1
        if count >= 4:
            break



=== XAI Model Interpretability Report ===

--- Review Sample #1 ---
Review Text: "I gave this film 10 not because it is a superbly consistent movie, but for it's pure ability to evoke emotions in its audience. The story of one-woman's-struggle-against-all-odds is an old cliché by now, but very few films have carried it off with so much warmth and sincerity as The Color Purple.<br /><br />It also showed a different side to the African-American experience - showing that after slaves were granted freedom many fell into the ways of the hated 'white man' and were abusive of their own people. I find this an important point as it goes against the portray-white-on-black-violence-and-win-an-Oscar trend.<br /><br />Also the acting performances are superb - especially Oprah who I now have a new found respect for.<br /><br />Well worth watching - but keep some tissue handy...."
Top driving tokens affecting this prediction according to SHAP:
  * Token: 'find ' | Impact: -0.0229 (🔴 Pushing NEGATIVE